In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os 

read data


In [31]:
data_folder = r'C:\Users\Eslam\Downloads\archive (6)'
os.chdir(data_folder)

print(f"cwd {os.getcwd()}\n")

#ensure csv
for file_name in os.listdir('.'):
    if file_name.endswith('.csv'):
            try:
                df = pd.read_csv(file_name)
                print(f"✅ file readed {file_name}")
                print(f"  num of rows and columns: {df.shape}")
                print(f"  sample of first 5 columns: {list(df.columns[:5])}") 
                print("-" * 40)
            except:
                print(f"❌ not readed: {file_name}")
            

cwd C:\Users\Eslam\Downloads\archive (6)

✅ file readed customers.csv
  num of rows and columns: (200, 8)
  sample of first 5 columns: ['customer_id', 'customer_name', 'customer_type', 'credit_terms_days', 'primary_freight_type']
----------------------------------------
✅ file readed delivery_events.csv
  num of rows and columns: (170820, 11)
  sample of first 5 columns: ['event_id', 'load_id', 'trip_id', 'event_type', 'facility_id']
----------------------------------------
✅ file readed drivers.csv
  num of rows and columns: (150, 12)
  sample of first 5 columns: ['driver_id', 'first_name', 'last_name', 'hire_date', 'termination_date']
----------------------------------------
✅ file readed driver_monthly_metrics.csv
  num of rows and columns: (4464, 9)
  sample of first 5 columns: ['driver_id', 'month', 'trips_completed', 'total_miles', 'total_revenue']
----------------------------------------
✅ file readed facilities.csv
  num of rows and columns: (50, 9)
  sample of first 5 columns:

In [32]:
customers = pd.read_csv('customers.csv')
delivery_events = pd.read_csv('delivery_events.csv')
driver_monthly_metrics = pd.read_csv('driver_monthly_metrics.csv')
drivers = pd.read_csv('drivers.csv')
facilities = pd.read_csv('facilities.csv')
fuel_purchases = pd.read_csv('fuel_purchases.csv')
loads = pd.read_csv('loads.csv')
maintenance_records = pd.read_csv('maintenance_records.csv')
routes = pd.read_csv('routes.csv')
safety_incidents = pd.read_csv('safety_incidents.csv')
trailers = pd.read_csv('trailers.csv')
trips = pd.read_csv('trips.csv')
truck_utilization_metrics = pd.read_csv('truck_utilization_metrics.csv')
trucks = pd.read_csv('trucks.csv')

customers cleaning

In [33]:
# unique vlaues per column and there datatypes
cols_to_convert = ['account_status', 'primary_freight_type', 'customer_type']
customers['contract_start_date']=pd.to_datetime(customers['contract_start_date'])
for col in cols_to_convert:
    customers[col] = customers[col].astype('category')
#handling data types
cols_to_convert = ['account_status', 'primary_freight_type', 'customer_type']
customers['contract_start_date']=pd.to_datetime(customers['contract_start_date'])
for col in cols_to_convert:
    customers[col] = customers[col].astype('category')
#view result 
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               200 non-null    str           
 1   customer_name             200 non-null    str           
 2   customer_type             200 non-null    category      
 3   credit_terms_days         200 non-null    int64         
 4   primary_freight_type      200 non-null    category      
 5   account_status            200 non-null    category      
 6   contract_start_date       200 non-null    datetime64[us]
 7   annual_revenue_potential  200 non-null    int64         
dtypes: category(3), datetime64[us](1), int64(2), str(2)
memory usage: 8.6 KB


delivery events cleaning

In [34]:
delivery_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 170820 entries, 0 to 170819
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   event_id            170820 non-null  str  
 1   load_id             170820 non-null  str  
 2   trip_id             170820 non-null  str  
 3   event_type          170820 non-null  str  
 4   facility_id         170820 non-null  str  
 5   scheduled_datetime  170820 non-null  str  
 6   actual_datetime     170820 non-null  str  
 7   detention_minutes   170820 non-null  int64
 8   on_time_flag        170820 non-null  bool 
 9   location_city       170820 non-null  str  
 10  location_state      170820 non-null  str  
dtypes: bool(1), int64(1), str(9)
memory usage: 13.2 MB


In [35]:
#handling data type tot date time
date_cols = ['scheduled_datetime', 'actual_datetime']
for col in date_cols:
    if col in delivery_events.columns:
        delivery_events[col] = pd.to_datetime(delivery_events[col], errors='coerce')

define on_time_flag probleme

In [36]:
delivery_events["delay_minutes"] = (
    delivery_events["actual_datetime"] - delivery_events["scheduled_datetime"]
).dt.total_seconds() / 60
#if df["delay_minutes"] =(-num)then 0
delivery_events["delay_minutes_clean"] = delivery_events["delay_minutes"].apply(lambda x: max(0, x))
delivery_events["correct_on_time_flag"] = delivery_events["actual_datetime"] <= delivery_events["scheduled_datetime"]

mismatched_rows = delivery_events[delivery_events["on_time_flag"] != delivery_events["correct_on_time_flag"]]

print(f"total differences between on_time_flag and correct_on_time_flag {len(mismatched_rows)}")
if len(mismatched_rows) > 0:
  print(
      mismatched_rows[
          [
              "scheduled_datetime",
              "actual_datetime",
              "delay_minutes",
              "on_time_flag",
              "correct_on_time_flag",
          ]
      ].head(10)
  )

total differences between on_time_flag and correct_on_time_flag 56866
           scheduled_datetime            actual_datetime  delay_minutes  \
10 2022-01-01 10:00:00.000000 2022-01-01 11:11:18.602953      71.310049   
11 2022-01-03 02:35:18.602953 2022-01-02 23:52:34.261273    -162.739028   
23 2022-01-02 08:54:16.218829 2022-01-02 09:00:21.031626       6.080213   
28 2022-01-01 07:00:00.000000 2022-01-01 07:47:13.542430      47.225707   
33 2022-01-02 16:12:02.952632 2022-01-02 17:29:23.553207      77.343343   
35 2022-01-02 13:30:26.571604 2022-01-02 11:18:22.416523    -132.069251   
38 2022-01-01 14:00:00.000000 2022-01-01 14:18:54.697099      18.911618   
40 2022-01-01 10:00:00.000000 2022-01-01 11:44:40.281617     104.671360   
42 2022-01-01 07:00:00.000000 2022-01-01 08:40:22.808653     100.380144   
44 2022-01-01 12:00:00.000000 2022-01-01 12:42:31.373576      42.522893   

    on_time_flag  correct_on_time_flag  
10          True                 False  
11         False      

In [37]:
#drop on time falg column
delivery_events.drop(columns=['on_time_flag'], inplace=True)

C:\Users\Eslam\AppData\Local\Temp\ipykernel_27896\3593932781.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  delivery_events.drop(columns=['on_time_flag'], inplace=True)


In [38]:
#order the columns
desired_order = [
    'event_id',
    'load_id',
    'trip_id',
    'event_type',
    'facility_id',
    'scheduled_datetime',
    'actual_datetime',
    'detention_minutes',
    'delay_minutes_clean',
    'correct_on_time_flag',
    'location_city',
    'location_state',
]

delivery_events = delivery_events[desired_order]

In [39]:
delivery_events

,event_id,load_id,trip_id,event_type,facility_id,scheduled_datetime,actual_datetime,detention_minutes,delay_minutes_clean,correct_on_time_flag,location_city,location_state
0,EVT00000001,LOAD00000001,TRIP00000001,Pickup,FAC00034,2022-01-01 18:00:00.000000,2022-01-01 20:58:55.918185,0,178.931970,False,Houston,TX
1,EVT00000002,LOAD00000001,TRIP00000001,Delivery,FAC00046,2022-01-02 23:10:55.918185,2022-01-02 21:30:22.142060,230,0.000000,True,Detroit,MI
2,EVT00000003,LOAD00000002,TRIP00000002,Pickup,FAC00015,2022-01-01 18:00:00.000000,2022-01-01 17:37:26.608430,62,0.000000,True,Kansas City,MO
3,EVT00000004,LOAD00000002,TRIP00000002,Delivery,FAC00050,2022-01-02 02:13:26.608430,2022-01-02 05:55:46.238257,67,222.327164,False,Indianapolis,IN
4,EVT00000005,LOAD00000003,TRIP00000003,Pickup,FAC00001,2022-01-01 08:00:00.000000,2022-01-01 07:28:02.169634,83,0.000000,True,Columbus,OH
...,...,...,...,...,...,...,...,...,...,...,...,...
170815,EVT00170816,LOAD00085408,TRIP00085408,Delivery,FAC00045,2025-01-01 17:57:42.158981,2025-01-01 15:19:00.254154,126,0.000000,True,Portland,OR
170816,EVT00170817,LOAD00085409,TRIP00085409,Pickup,FAC00025,2024-12-31 13:00:00.000000,2024-12-31 16:37:42.056974,13,217.700950,False,Philadelphia,PA
170817,EVT00170818,LOAD00085409,TRIP00085409,Delivery,FAC00001,2025-01-02 16:31:42.056974,2025-01-02 21:15:48.779380,81,284.112040,False,Seattle,WA
170818,EVT00170819,LOAD00085410,TRIP00085410,Pickup,FAC00027,2024-12-31 06:00:00.000000,2024-12-31 05:30:59.579843,48,0.000000,True,Dallas,TX


clean drivers

In [40]:
#combine first name with last name in full name
drivers['first_name'] = drivers['first_name'].str.strip().str.title()
drivers['last_name'] = drivers['last_name'].str.strip().str.title()


drivers['full_name'] = drivers['first_name'] + ' ' + drivers['last_name']

print(drivers[['first_name', 'last_name', 'full_name']].sample(5))
#drop first name with last name
drivers = drivers.drop(columns=['first_name', 'last_name'], errors='ignore')

print(drivers.columns)

    first_name last_name          full_name
113     Thomas    Wilson      Thomas Wilson
88       David    Thomas       David Thomas
106    Charles     Jones      Charles Jones
76    Patricia  Anderson  Patricia Anderson
57       David    Miller       David Miller
Index(['driver_id', 'hire_date', 'termination_date', 'license_number',
       'license_state', 'date_of_birth', 'home_terminal', 'employment_status',
       'cdl_class', 'years_experience', 'full_name'],
      dtype='str')


In [41]:
# reorder deivers data frame
desired_order = [
    'driver_id',
    'full_name',
    'hire_date',
    'termination_date',
    'license_number',
    'license_state',
    'date_of_birth',
    'home_terminal',
    'employment_status',
    'cdl_class',
    'years_experience'
]

drivers = drivers[desired_order]

In [42]:
drivers.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   driver_id          150 non-null    str  
 1   full_name          150 non-null    str  
 2   hire_date          150 non-null    str  
 3   termination_date   26 non-null     str  
 4   license_number     150 non-null    str  
 5   license_state      150 non-null    str  
 6   date_of_birth      150 non-null    str  
 7   home_terminal      150 non-null    str  
 8   employment_status  150 non-null    str  
 9   cdl_class          150 non-null    str  
 10  years_experience   150 non-null    int64
dtypes: int64(1), str(10)
memory usage: 13.0 KB


In [43]:
date_columns = ['hire_date', 'termination_date', 'date_of_birth']

for col in date_columns:
    drivers[col] = pd.to_datetime(drivers[col], errors='coerce')



In [44]:
# years of life - experience
drivers['age_at_career_start'] = (pd.to_datetime('2024-12-31') - pd.to_datetime(drivers['date_of_birth'])).dt.days / 365.25 - drivers['years_experience']


In [45]:
invalid_drivers = drivers[drivers['age_at_career_start'] < 18]

invalid_count = len(invalid_drivers)
total_drivers = len(drivers)
invalid_percentage = (invalid_count / total_drivers) * 100

print(f" liers drivers :{invalid_count}")
print(f"total drivers : {total_drivers} \n")
print(f"percentage od liers drivers {invalid_percentage:.2f}% ")

 liers drivers :21
total drivers : 150 

percentage od liers drivers 14.00% 


In [46]:
invalid_drivers

,driver_id,full_name,hire_date,termination_date,license_number,license_state,date_of_birth,home_terminal,employment_status,cdl_class,years_experience,age_at_career_start
3,DRV00004,Barbara Brown,2013-09-08,NaT,DL735540030,WA,1995-02-06,Denver,Active,A,19,10.900068
5,DRV00006,Robert Taylor,2019-04-04,NaT,DL811890569,NC,1990-05-02,Portland,Active,A,23,11.666667
11,DRV00012,Robert Moore,2013-09-12,NaT,DL595744935,MO,1994-04-21,Nashville,Active,A,25,5.696783
18,DRV00019,Robert Jackson,2020-12-14,NaT,DL588229686,TX,1994-08-15,Seattle,Active,A,20,10.379192
34,DRV00035,Michael Taylor,2018-11-19,NaT,DL155926488,MI,1989-02-11,Atlanta,Active,A,19,16.885010
38,DRV00039,Thomas Martinez,2013-06-29,NaT,DL173678414,TX,1994-08-14,Columbus,Active,A,22,8.381930
41,DRV00042,Richard Hernandez,2014-03-05,2015-09-07,DL143226844,TN,1991-11-15,Los Angeles,Terminated,A,21,12.127995
43,DRV00044,Richard Davis,2018-03-10,NaT,DL133318206,NC,1995-03-10,Atlanta,Active,A,24,5.812457
48,DRV00049,John Smith,2018-03-25,NaT,DL263619298,OK,1995-07-03,Memphis,Active,A,24,5.497604
55,DRV00056,William Wilson,2018-02-23,NaT,DL334037646,MI,1989-09-05,Oklahoma City,Active,A,20,15.321013


cleaning facilities

In [47]:
facilities['is_24_7'] = facilities['operating_hours'].str.contains('24/7').astype(bool)

#new data frame time_split for columns that have -
time_split = facilities['operating_hours'].str.split('-', expand=True)
facilities['open_time'] = time_split[0].str.strip()
facilities['close_time'] = time_split[1].str.strip()
    
facilities.loc[facilities['is_24_7'] == 1, 'open_time'] = '12:00 AM'
facilities.loc[facilities['is_24_7'] == 1, 'close_time'] = '11:59 PM'
#drop messy column
facilities = facilities.drop(columns=['operating_hours'])

In [48]:
facilities.sample(5)

,facility_id,facility_name,facility_type,city,state,latitude,longitude,dock_doors,is_24_7,open_time,close_time
4,FAC00005,Indianapolis Hub,Distribution Center,Indianapolis,IN,39.7684,-86.1581,56,False,8AM,5PM
5,FAC00006,Atlanta Warehouse,Distribution Center,Atlanta,GA,33.7490,-84.3880,149,False,8AM,5PM
3,FAC00004,Chicago Distribution Center,Distribution Center,Chicago,IL,41.8781,-87.6298,38,False,8AM,5PM
38,FAC00039,Charlotte Distribution Center,Warehouse,Charlotte,NC,35.2271,-80.8431,47,False,7AM,7PM
24,FAC00025,Oklahoma City Terminal,Cross-Dock,Oklahoma City,OK,35.4676,-97.5164,102,True,12:00 AM,11:59 PM


In [49]:
facilities.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   facility_id    50 non-null     str    
 1   facility_name  50 non-null     str    
 2   facility_type  50 non-null     str    
 3   city           50 non-null     str    
 4   state          50 non-null     str    
 5   latitude       50 non-null     float64
 6   longitude      50 non-null     float64
 7   dock_doors     50 non-null     int64  
 8   is_24_7        50 non-null     bool   
 9   open_time      50 non-null     str    
 10  close_time     50 non-null     str    
dtypes: bool(1), float64(2), int64(1), str(7)
memory usage: 4.1 KB


clean fuel_purchases

In [50]:
fuel_purchases['purchase_date'] = pd.to_datetime(fuel_purchases['purchase_date'])

print(fuel_purchases.dtypes)

fuel_purchase_id               str
trip_id                        str
truck_id                       str
driver_id                      str
purchase_date       datetime64[us]
location_city                  str
location_state                 str
gallons                    float64
price_per_gallon           float64
total_cost                 float64
fuel_card_number               str
dtype: object


In [51]:
fuel_purchases.info()

<class 'pandas.DataFrame'>
RangeIndex: 196442 entries, 0 to 196441
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   fuel_purchase_id  196442 non-null  str           
 1   trip_id           196442 non-null  str           
 2   truck_id          192562 non-null  str           
 3   driver_id         192454 non-null  str           
 4   purchase_date     196442 non-null  datetime64[us]
 5   location_city     196442 non-null  str           
 6   location_state    196442 non-null  str           
 7   gallons           196442 non-null  float64       
 8   price_per_gallon  196442 non-null  float64       
 9   total_cost        196442 non-null  float64       
 10  fuel_card_number  196442 non-null  str           
dtypes: datetime64[us](1), float64(3), str(7)
memory usage: 16.5 MB


In [52]:
arget_cols = ['truck_id', 'driver_id']

#handle missing values
fuel_purchases[arget_cols] = fuel_purchases[arget_cols].fillna('unknown')

print("num after remove nulls")
print(fuel_purchases[arget_cols].isnull().sum())

num after remove nulls
truck_id     0
driver_id    0
dtype: int64


clean safety_incidents

In [53]:
arget_cols = ['truck_id', 'driver_id']

print("num of nulls")
print(safety_incidents[arget_cols].isnull().sum())

#handle missing values
safety_incidents[arget_cols] = safety_incidents[arget_cols].fillna('unknown')

print("num after remove nulls")
print(safety_incidents[arget_cols].isnull().sum())

num of nulls
truck_id     1
driver_id    1
dtype: int64
num after remove nulls
truck_id     0
driver_id    0
dtype: int64


In [54]:
#convert datetime
safety_incidents['incident_date'] = pd.to_datetime(safety_incidents['incident_date'])
safety_incidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   incident_id          170 non-null    str           
 1   trip_id              170 non-null    str           
 2   truck_id             170 non-null    str           
 3   driver_id            170 non-null    str           
 4   incident_date        170 non-null    datetime64[us]
 5   incident_type        170 non-null    str           
 6   location_city        170 non-null    str           
 7   location_state       170 non-null    str           
 8   at_fault_flag        170 non-null    bool          
 9   injury_flag          170 non-null    bool          
 10  vehicle_damage_cost  170 non-null    float64       
 11  cargo_damage_cost    170 non-null    float64       
 12  claim_amount         170 non-null    float64       
 13  preventable_flag     170 non-null    bool     

clean trips

In [55]:
trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 85410 entries, 0 to 85409
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trip_id                85410 non-null  str    
 1   load_id                85410 non-null  str    
 2   driver_id              83696 non-null  str    
 3   truck_id               83738 non-null  str    
 4   trailer_id             83730 non-null  str    
 5   dispatch_date          85410 non-null  str    
 6   actual_distance_miles  85410 non-null  int64  
 7   actual_duration_hours  85410 non-null  float64
 8   fuel_gallons_used      85410 non-null  float64
 9   average_mpg            85410 non-null  float64
 10  idle_time_hours        85410 non-null  float64
 11  trip_status            85410 non-null  str    
dtypes: float64(4), int64(1), str(7)
memory usage: 7.8 MB


In [56]:
arget_cols = ['driver_id', 'truck_id', 'trailer_id']

print("num of nulls")
print(trips[arget_cols].isnull().sum())

#handle missing values
trips[arget_cols] = trips[arget_cols].fillna('unknown')

print("num after remove nulls")
print(trips[arget_cols].isnull().sum())

num of nulls
driver_id     1714
truck_id      1672
trailer_id    1680
dtype: int64
num after remove nulls
driver_id     0
truck_id      0
trailer_id    0
dtype: int64


In [57]:
#convert to datetime
trips['dispatch_date'] = pd.to_datetime(trips['dispatch_date'])
trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 85410 entries, 0 to 85409
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   trip_id                85410 non-null  str           
 1   load_id                85410 non-null  str           
 2   driver_id              85410 non-null  str           
 3   truck_id               85410 non-null  str           
 4   trailer_id             85410 non-null  str           
 5   dispatch_date          85410 non-null  datetime64[us]
 6   actual_distance_miles  85410 non-null  int64         
 7   actual_duration_hours  85410 non-null  float64       
 8   fuel_gallons_used      85410 non-null  float64       
 9   average_mpg            85410 non-null  float64       
 10  idle_time_hours        85410 non-null  float64       
 11  trip_status            85410 non-null  str           
dtypes: datetime64[us](1), float64(4), int64(1), str(6)
memory usage: 7.8 MB

In [58]:
from sqlalchemy import create_engine

#sql info
server_name = r'.'
database_name = 'Logistics'

connection_string = (
    f"mssql+pyodbc://@{server_name}/{database_name}"
    f"?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

engine = create_engine(connection_string)

dfs_dict = {
    'customers': customers,
    'delivery_events': delivery_events,
    'driver_monthly_metrics': driver_monthly_metrics,
    'drivers': drivers,
    'facilities': facilities,
    'fuel_purchases': fuel_purchases,
    'loads': loads,
    'maintenance_records': maintenance_records,
    'routes': routes,
    'safety_incidents': safety_incidents,
    'trailers': trailers,
    'trips': trips,
    'truck_utilization_metrics': truck_utilization_metrics,
    'trucks': trucks
}

for table_name, df in dfs_dict.items():
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists='replace',
        index=False
    )
    print(f"[{table_name}]uploded ")


ProgrammingError: (pyodbc.ProgrammingError) ('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Could not drop object 'customers' because it is referenced by a FOREIGN KEY constraint. (3726) (SQLExecDirectW)")
[SQL: 
DROP TABLE customers]
(Background on this error at: https://sqlalche.me/e/21/f405)

In [ ]:
from sqlalchemy import text

# كود الـ SQL الشامل لبناء المفاتيح والعلاقات
modeling_sql = """
-- ==========================================
-- أولاً: تعديل الأعمدة وإضافة المفاتيح الأساسية (Primary Keys)
-- ==========================================
ALTER TABLE customers ALTER COLUMN customer_id NVARCHAR(50) NOT NULL;
ALTER TABLE customers ADD CONSTRAINT PK_customers PRIMARY KEY (customer_id);

ALTER TABLE routes ALTER COLUMN route_id NVARCHAR(50) NOT NULL;
ALTER TABLE routes ADD CONSTRAINT PK_routes PRIMARY KEY (route_id);

ALTER TABLE drivers ALTER COLUMN driver_id NVARCHAR(50) NOT NULL;
ALTER TABLE drivers ADD CONSTRAINT PK_drivers PRIMARY KEY (driver_id);

ALTER TABLE trucks ALTER COLUMN truck_id NVARCHAR(50) NOT NULL;
ALTER TABLE trucks ADD CONSTRAINT PK_trucks PRIMARY KEY (truck_id);

ALTER TABLE trailers ALTER COLUMN trailer_id NVARCHAR(50) NOT NULL;
ALTER TABLE trailers ADD CONSTRAINT PK_trailers PRIMARY KEY (trailer_id);

ALTER TABLE facilities ALTER COLUMN facility_id NVARCHAR(50) NOT NULL;
ALTER TABLE facilities ADD CONSTRAINT PK_facilities PRIMARY KEY (facility_id);

ALTER TABLE loads ALTER COLUMN load_id NVARCHAR(50) NOT NULL;
ALTER TABLE loads ADD CONSTRAINT PK_loads PRIMARY KEY (load_id);

ALTER TABLE trips ALTER COLUMN trip_id NVARCHAR(50) NOT NULL;
ALTER TABLE trips ADD CONSTRAINT PK_trips PRIMARY KEY (trip_id);

ALTER TABLE fuel_purchases ALTER COLUMN fuel_purchase_id NVARCHAR(50) NOT NULL;
ALTER TABLE fuel_purchases ADD CONSTRAINT PK_fuel_purchases PRIMARY KEY (fuel_purchase_id);

ALTER TABLE maintenance_records ALTER COLUMN maintenance_id NVARCHAR(50) NOT NULL;
ALTER TABLE maintenance_records ADD CONSTRAINT PK_maintenance_records PRIMARY KEY (maintenance_id);

ALTER TABLE delivery_events ALTER COLUMN event_id NVARCHAR(50) NOT NULL;
ALTER TABLE delivery_events ADD CONSTRAINT PK_delivery_events PRIMARY KEY (event_id);

ALTER TABLE safety_incidents ALTER COLUMN incident_id NVARCHAR(50) NOT NULL;
ALTER TABLE safety_incidents ADD CONSTRAINT PK_safety_incidents PRIMARY KEY (incident_id);

-- ==========================================
-- ثانياً: توحيد أنواع بيانات المفاتيح الأجنبية (Foreign Keys)
-- ==========================================
ALTER TABLE loads ALTER COLUMN customer_id NVARCHAR(50) NULL;
ALTER TABLE loads ALTER COLUMN route_id NVARCHAR(50) NULL;

ALTER TABLE trips ALTER COLUMN load_id NVARCHAR(50) NULL;
ALTER TABLE trips ALTER COLUMN driver_id NVARCHAR(50) NULL;
ALTER TABLE trips ALTER COLUMN truck_id NVARCHAR(50) NULL;
ALTER TABLE trips ALTER COLUMN trailer_id NVARCHAR(50) NULL;

ALTER TABLE fuel_purchases ALTER COLUMN trip_id NVARCHAR(50) NULL;
ALTER TABLE fuel_purchases ALTER COLUMN truck_id NVARCHAR(50) NULL;
ALTER TABLE fuel_purchases ALTER COLUMN driver_id NVARCHAR(50) NULL;

ALTER TABLE maintenance_records ALTER COLUMN truck_id NVARCHAR(50) NULL;

ALTER TABLE delivery_events ALTER COLUMN load_id NVARCHAR(50) NULL;
ALTER TABLE delivery_events ALTER COLUMN trip_id NVARCHAR(50) NULL;
ALTER TABLE delivery_events ALTER COLUMN facility_id NVARCHAR(50) NULL;

ALTER TABLE safety_incidents ALTER COLUMN trip_id NVARCHAR(50) NULL;
ALTER TABLE safety_incidents ALTER COLUMN truck_id NVARCHAR(50) NULL;
ALTER TABLE safety_incidents ALTER COLUMN driver_id NVARCHAR(50) NULL;

-- ==========================================
-- ثالثاً: بناء العلاقات (Relationships) 
-- ==========================================
ALTER TABLE loads WITH NOCHECK ADD CONSTRAINT FK_loads_customers FOREIGN KEY (customer_id) REFERENCES customers(customer_id);
ALTER TABLE loads WITH NOCHECK ADD CONSTRAINT FK_loads_routes FOREIGN KEY (route_id) REFERENCES routes(route_id);

ALTER TABLE trips WITH NOCHECK ADD CONSTRAINT FK_trips_loads FOREIGN KEY (load_id) REFERENCES loads(load_id);
ALTER TABLE trips WITH NOCHECK ADD CONSTRAINT FK_trips_drivers FOREIGN KEY (driver_id) REFERENCES drivers(driver_id);
ALTER TABLE trips WITH NOCHECK ADD CONSTRAINT FK_trips_trucks FOREIGN KEY (truck_id) REFERENCES trucks(truck_id);
ALTER TABLE trips WITH NOCHECK ADD CONSTRAINT FK_trips_trailers FOREIGN KEY (trailer_id) REFERENCES trailers(trailer_id);

ALTER TABLE fuel_purchases WITH NOCHECK ADD CONSTRAINT FK_fuel_trips FOREIGN KEY (trip_id) REFERENCES trips(trip_id);
ALTER TABLE fuel_purchases WITH NOCHECK ADD CONSTRAINT FK_fuel_trucks FOREIGN KEY (truck_id) REFERENCES trucks(truck_id);
ALTER TABLE fuel_purchases WITH NOCHECK ADD CONSTRAINT FK_fuel_drivers FOREIGN KEY (driver_id) REFERENCES drivers(driver_id);

ALTER TABLE maintenance_records WITH NOCHECK ADD CONSTRAINT FK_maintenance_trucks FOREIGN KEY (truck_id) REFERENCES trucks(truck_id);

ALTER TABLE delivery_events WITH NOCHECK ADD CONSTRAINT FK_delivery_loads FOREIGN KEY (load_id) REFERENCES loads(load_id);
ALTER TABLE delivery_events WITH NOCHECK ADD CONSTRAINT FK_delivery_trips FOREIGN KEY (trip_id) REFERENCES trips(trip_id);
ALTER TABLE delivery_events WITH NOCHECK ADD CONSTRAINT FK_delivery_facilities FOREIGN KEY (facility_id) REFERENCES facilities(facility_id);

ALTER TABLE safety_incidents WITH NOCHECK ADD CONSTRAINT FK_safety_trips FOREIGN KEY (trip_id) REFERENCES trips(trip_id);
ALTER TABLE safety_incidents WITH NOCHECK ADD CONSTRAINT FK_safety_trucks FOREIGN KEY (truck_id) REFERENCES trucks(truck_id);
ALTER TABLE safety_incidents WITH NOCHECK ADD CONSTRAINT FK_safety_drivers FOREIGN KEY (driver_id) REFERENCES drivers(driver_id);
"""

# تنفيذ الكود على قاعدة البيانات
try:
    with engine.begin() as conn:
        conn.execute(text(modeling_sql))
    print("تم بناء الموديل (Primary & Foreign Keys) وتوصيل الجداول ببعضها بنجاح!")
except Exception as e:
    print(f"حدث خطأ أثناء بناء الموديل: {e}")

In [ ]:
# استعلام للتأكد من وجود العلاقات في قاعدة البيانات
check_sql = """
SELECT 
    fk.name AS [Foreign Key],
    tp.name AS [Fact Table],
    tr.name AS [Dim Table]
FROM sys.foreign_keys fk
INNER JOIN sys.tables tp ON fk.parent_object_id = tp.object_id
INNER JOIN sys.tables tr ON fk.referenced_object_id = tr.object_id;
"""

# قراءة النتيجة في DataFrame وعرضها
df_check = pd.read_sql(check_sql, engine)

if df_check.empty:
    print("مفيش أي علاقات اتعملت! راجع الكود اللي فات.")
else:
    print(f"عاش! الموديلنج نجح وتم إنشاء {len(df_check)} علاقة.")
    display(df_check) # هيعرضلك جدول بكل العلاقات اللي اتعملت

عاش! الموديلنج نجح وتم إنشاء 9 علاقة.


,Foreign Key,Fact Table,Dim Table
0,FK_loads_customers,loads,customers
1,FK_trips_drivers,trips,drivers
2,FK_trips_loads,trips,loads
3,FK_loads_routes,loads,routes
4,FK_trips_trailers,trips,trailers
5,FK_fuel_trips,fuel_purchases,trips
6,FK_delivery_trips,delivery_events,trips
7,FK_safety_trips,safety_incidents,trips
8,FK_trips_trucks,trips,trucks


In [ ]:
from sqlalchemy import text

# كود لإضافة الـ 7 علاقات الناقصة فقط (مع تخطي الأخطاء)
missing_relations_sql = """
-- 1. مسح العلاقات دي لو كانت متعلقة عشان متعملش إيرور
ALTER TABLE fuel_purchases DROP CONSTRAINT IF EXISTS FK_fuel_trucks;
ALTER TABLE fuel_purchases DROP CONSTRAINT IF EXISTS FK_fuel_drivers;
ALTER TABLE maintenance_records DROP CONSTRAINT IF EXISTS FK_maintenance_trucks;
ALTER TABLE delivery_events DROP CONSTRAINT IF EXISTS FK_delivery_loads;
ALTER TABLE delivery_events DROP CONSTRAINT IF EXISTS FK_delivery_facilities;
ALTER TABLE safety_incidents DROP CONSTRAINT IF EXISTS FK_safety_trucks;
ALTER TABLE safety_incidents DROP CONSTRAINT IF EXISTS FK_safety_drivers;

-- 2. إضافة الـ 7 علاقات الناقصين
ALTER TABLE fuel_purchases WITH NOCHECK ADD CONSTRAINT FK_fuel_trucks FOREIGN KEY (truck_id) REFERENCES trucks(truck_id);
ALTER TABLE fuel_purchases WITH NOCHECK ADD CONSTRAINT FK_fuel_drivers FOREIGN KEY (driver_id) REFERENCES drivers(driver_id);

ALTER TABLE maintenance_records WITH NOCHECK ADD CONSTRAINT FK_maintenance_trucks FOREIGN KEY (truck_id) REFERENCES trucks(truck_id);

ALTER TABLE delivery_events WITH NOCHECK ADD CONSTRAINT FK_delivery_loads FOREIGN KEY (load_id) REFERENCES loads(load_id);
ALTER TABLE delivery_events WITH NOCHECK ADD CONSTRAINT FK_delivery_facilities FOREIGN KEY (facility_id) REFERENCES facilities(facility_id);

ALTER TABLE safety_incidents WITH NOCHECK ADD CONSTRAINT FK_safety_trucks FOREIGN KEY (truck_id) REFERENCES trucks(truck_id);
ALTER TABLE safety_incidents WITH NOCHECK ADD CONSTRAINT FK_safety_drivers FOREIGN KEY (driver_id) REFERENCES drivers(driver_id);
"""

# تنفيذ الكود
try:
    with engine.begin() as conn:
        conn.execute(text(missing_relations_sql))
    print("عاش جداً! تم إضافة الـ 7 علاقات الناقصة بنجاح.")
except Exception as e:
    print(f"حدث خطأ: {e}")

عاش جداً! تم إضافة الـ 7 علاقات الناقصة بنجاح.


In [ ]:
import pandas as pd

# استعلام للتأكد من كل العلاقات الموجودة حالياً في قاعدة البيانات
check_sql = """
SELECT 
    fk.name AS [Foreign Key],
    tp.name AS [Fact Table],
    tr.name AS [Dim Table]
FROM sys.foreign_keys fk
INNER JOIN sys.tables tp ON fk.parent_object_id = tp.object_id
INNER JOIN sys.tables tr ON fk.referenced_object_id = tr.object_id
ORDER BY tp.name;
"""

# قراءة النتيجة في DataFrame وعرضها
df_check = pd.read_sql(check_sql, engine)

print(f"إجمالي العلاقات الموجودة حالياً: {len(df_check)} علاقة من أصل 16")
print("-" * 50)

# عرض الجدول بالكامل
pd.set_option('display.max_rows', None) # عشان يظهر الجدول كله بدون اختصار
display(df_check)

إجمالي العلاقات الموجودة حالياً: 16 علاقة من أصل 16
--------------------------------------------------


,Foreign Key,Fact Table,Dim Table
0,FK_delivery_facilities,delivery_events,facilities
1,FK_delivery_loads,delivery_events,loads
2,FK_delivery_trips,delivery_events,trips
3,FK_fuel_trucks,fuel_purchases,trucks
4,FK_fuel_trips,fuel_purchases,trips
5,FK_fuel_drivers,fuel_purchases,drivers
6,FK_loads_customers,loads,customers
7,FK_loads_routes,loads,routes
8,FK_maintenance_trucks,maintenance_records,trucks
9,FK_safety_trucks,safety_incidents,trucks


In [ ]:
from sqlalchemy import text

# كود SQL لربط جداول الـ Metrics المعزولة
metrics_relations_sql = """
-- 1. توحيد نوع البيانات لضمان عدم حدوث خطأ 1778
ALTER TABLE driver_monthly_metrics ALTER COLUMN driver_id NVARCHAR(50) NULL;
ALTER TABLE truck_utilization_metrics ALTER COLUMN truck_id NVARCHAR(50) NULL;

-- 2. مسح أي قيود قديمة إن وجدت
ALTER TABLE driver_monthly_metrics DROP CONSTRAINT IF EXISTS FK_driver_metrics_drivers;
ALTER TABLE truck_utilization_metrics DROP CONSTRAINT IF EXISTS FK_truck_metrics_trucks;

-- 3. إنشاء العلاقات وربطها بجداول الأبعاد الأساسية
ALTER TABLE driver_monthly_metrics WITH NOCHECK 
ADD CONSTRAINT FK_driver_metrics_drivers FOREIGN KEY (driver_id) REFERENCES drivers(driver_id);

ALTER TABLE truck_utilization_metrics WITH NOCHECK 
ADD CONSTRAINT FK_truck_metrics_trucks FOREIGN KEY (truck_id) REFERENCES trucks(truck_id);
"""

# تنفيذ الكود
try:
    with engine.begin() as conn:
        conn.execute(text(metrics_relations_sql))
    print("تم ربط جدولين الـ Metrics (السائقين والشاحنات) بنجاح! الموديل كده اكتمل 100%.")
except Exception as e:
    print(f"حدث خطأ: {e}")

تم ربط جدولين الـ Metrics (السائقين والشاحنات) بنجاح! الموديل كده اكتمل 100%.


In [ ]:
Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned